# Readability as Circuit Resistance - Demo Notebook

## Experiment Overview

This notebook implements a novel method for scoring text readability using **graph-based effective resistance** (Kirchhoff index).

### Method
1. Convert text to discourse graph using sentence word overlap for similarity
2. Add edges based on word overlap similarity (Jaccard similarity > threshold)
3. Compute effective graph resistance (Kirchhoff index) using NetworkX
4. Use resistance score as readability metric (lower = more connected = more readable)

### Baseline
- Traditional Flesch-Kincaid Grade Level formula

### Expected Output
The notebook processes text examples and outputs predictions from both methods in `exp_gen_sol_out.json` schema format with `predict_our_method` and `predict_baseline` fields.

In [ ]:
# Install dependencies - Colab compatible pattern
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install everywhere)
_pip('loguru')
_pip('textstat')
_pip('psutil')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'scikit-learn==1.6.1', 'matplotlib==3.10.0', 'networkx==3.6.1')

In [ ]:
# Imports - copied from original method.py
from loguru import logger
from pathlib import Path
import json
import sys
import gc
import resource
import psutil
import numpy as np
import networkx as nx
import textstat
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr

# Additional imports for notebook visualization
import matplotlib.pyplot as plt

# Setup logging
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
# Data loading helper - GitHub URL with local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-6d018e-readability-as-circuit-resistance-a-nove/main/round-2/experiment-1/demo/mini_demo_data.json"

import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
# Load the demo data
data = load_data()
print(f"Loaded {len(data.get('datasets', []))} datasets")
for dataset in data.get('datasets', []):
    print(f"  - {dataset.get('dataset')}: {len(dataset.get('examples', []))} examples")

## Configuration

Set tunable parameters to minimum values for quick demo execution. These can be scaled up once the notebook runs successfully.

In [ ]:
# Configuration - set to ABSOLUTE MINIMUM values for demo
SIMILARITY_THRESHOLD = 0.1  # Jaccard similarity threshold for graph edges
MAX_SENTENCES = 100  # Maximum sentences to process per text

# For demo: process all examples (already small dataset)
MAX_EXAMPLES = None  # Set to small number (e.g., 3) to process fewer examples

print(f"Config: SIMILARITY_THRESHOLD={SIMILARITY_THRESHOLD}, MAX_SENTENCES={MAX_SENTENCES}")

## Helper Functions

These functions implement the core method:
1. `text_to_sentences()` - Split text into sentences
2. `get_word_set()` - Extract words from a sentence
3. `build_word_overlap_graph()` - Build discourse graph with word overlap edges
4. `compute_effective_resistance()` - Calculate Kirchhoff index
5. `compute_flesch_kincaid_grade()` - Baseline readability score

In [ ]:
# Helper functions - copied from original method.py with minimal changes

def text_to_sentences(text: str, max_sentences: int = MAX_SENTENCES) -> list:
    """Split text into sentences using simple rules."""
    import re
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences[:max_sentences]


def get_word_set(sentence: str) -> set:
    """Get set of lowercase words from a sentence."""
    import re
    words = re.findall(r'\b[a-zA-Z]+\b', sentence.lower())
    return set(words)


def build_word_overlap_graph(sentences: list, threshold: float = SIMILARITY_THRESHOLD) -> nx.Graph:
    """Build a graph where edges connect sentences with word overlap."""
    if len(sentences) < 2:
        G = nx.Graph()
        if sentences:
            G.add_node(0, sentence=sentences[0])
        return G
    
    G = nx.Graph()
    
    for i, sent in enumerate(sentences):
        G.add_node(i, sentence=sent[:50])
    
    word_sets = [get_word_set(sent) for sent in sentences]
    
    for i in range(len(sentences)):
        for j in range(i + 1, len(sentences)):
            set_i = word_sets[i]
            set_j = word_sets[j]
            
            if len(set_i) == 0 and len(set_j) == 0:
                similarity = 1.0
            elif len(set_i) == 0 or len(set_j) == 0:
                similarity = 0.0
            else:
                intersection = len(set_i & set_j)
                union = len(set_i | set_j)
                similarity = intersection / union if union > 0 else 0.0
            
            if similarity > threshold:
                G.add_edge(i, j, weight=similarity)
    
    if len(sentences) > 1:
        for i in range(len(sentences) - 1):
            if not G.has_edge(i, i + 1):
                G.add_edge(i, i + 1, weight=0.1)
    
    return G


def compute_effective_resistance(G: nx.Graph) -> float:
    """Compute effective graph resistance (Kirchhoff index)."""
    if G.number_of_nodes() < 2:
        return 0.0
    
    try:
        resistance = nx.effective_graph_resistance(G)
        return float(resistance)
    except Exception as e:
        logger.warning(f"Error computing resistance: {e}")
        n = G.number_of_nodes()
        if nx.is_connected(G):
            avg_path = nx.average_shortest_path_length(G)
            return float(avg_path * n)
        else:
            return float(n * 10)


def compute_flesch_kincaid_grade(text: str) -> float:
    """Compute Flesch-Kincaid Grade Level (baseline)."""
    try:
        score = textstat.flesch_kincaid_grade(text)
        return float(score)
    except Exception as e:
        logger.warning(f"Error computing Flesch-Kincaid: {e}")
        return 0.0

## Process Examples

Run both methods (graph resistance and Flesch-Kincaid) on each example and collect predictions.

In [ ]:
# Process each dataset
results = {"metadata": data.get("metadata", {}), "datasets": []}

for dataset_idx, dataset in enumerate(data.get("datasets", [])):
    dataset_name = dataset.get("dataset", f"dataset_{dataset_idx}")
    examples = dataset.get("examples", [])
    
    if MAX_EXAMPLES is not None:
        examples = examples[:MAX_EXAMPLES]
    
    print(f"\nProcessing dataset: {dataset_name} ({len(examples)} examples)")
    processed_examples = []
    
    for i, example in enumerate(examples):
        print(f"  Processing example {i+1}/{len(examples)}")
        text = example.get("input", "")
        
        try:
            sentences = text_to_sentences(text)
            graph = build_word_overlap_graph(sentences)
            resistance_score = compute_effective_resistance(graph)
            n = len(sentences)
            normalized_resistance = resistance_score / max(n, 1)
            example["predict_our_method"] = str(normalized_resistance)
        except Exception as e:
            logger.error(f"Error in our method for example {i}: {e}")
            example["predict_our_method"] = "0.0"
        
        try:
            fk_score = compute_flesch_kincaid_grade(text)
            example["predict_baseline"] = str(fk_score)
        except Exception as e:
            logger.error(f"Error in baseline for example {i}: {e}")
            example["predict_baseline"] = "0.0"
        
        processed_examples.append(example)
    
    results["datasets"].append({
        "dataset": dataset_name,
        "examples": processed_examples
    })
    print(f"Completed dataset: {dataset_name}")

## Evaluate Results

Compute MAE, RMSE, and Pearson correlation for both methods compared to ground truth.

In [ ]:
def compute_metrics(results):
    """Compute evaluation metrics."""
    for dataset in results.get("datasets", []):
        dataset_name = dataset.get("dataset", "unknown")
        examples = dataset.get("examples", [])
        
        if not examples:
            continue
        
        our_preds = []
        baseline_preds = []
        ground_truth = []
        
        for ex in examples:
            try:
                output = ex.get("output", "")
                gt = float(output)
                our_pred = float(ex.get("predict_our_method", 0.0))
                baseline_pred = float(ex.get("predict_baseline", 0.0))
                ground_truth.append(gt)
                our_preds.append(our_pred)
                baseline_preds.append(baseline_pred)
            except (ValueError, TypeError) as e:
                logger.warning(f"Could not parse: {e}")
        
        if len(ground_truth) > 0:
            our_preds = np.array(our_preds)
            baseline_preds = np.array(baseline_preds)
            ground_truth = np.array(ground_truth)
            
            print(f"\n=== Metrics for {dataset_name} ===")
            
            try:
                our_mae = mean_absolute_error(ground_truth, our_preds)
                our_rmse = np.sqrt(mean_squared_error(ground_truth, our_preds))
                if len(ground_truth) > 1:
                    our_corr, _ = pearsonr(ground_truth, our_preds)
                else:
                    our_corr = 0.0
                print(f"Our Method (Resistance):")
                print(f"  MAE: {our_mae:.4f}")
                print(f"  RMSE: {our_rmse:.4f}")
                print(f"  Pearson r: {our_corr:.4f}")
            except Exception as e:
                print(f"Metrics failed for our method: {e}")
            
            try:
                baseline_mae = mean_absolute_error(ground_truth, baseline_preds)
                baseline_rmse = np.sqrt(mean_squared_error(ground_truth, baseline_preds))
                if len(ground_truth) > 1:
                    baseline_corr, _ = pearsonr(ground_truth, baseline_preds)
                else:
                    baseline_corr = 0.0
                print(f"Baseline (Flesch-Kincaid):")
                print(f"  MAE: {baseline_mae:.4f}")
                print(f"  RMSE: {baseline_rmse:.4f}")
                print(f"  Pearson r: {baseline_corr:.4f}")
            except Exception as e:
                print(f"Metrics failed for baseline: {e}")


compute_metrics(results)

## Visualize Results

Compare predictions from both methods against ground truth.

In [ ]:
def visualize_results(results):
    """Create visualization comparing predictions to ground truth."""
    for dataset in results.get("datasets", []):
        dataset_name = dataset.get("dataset", "unknown")
        examples = dataset.get("examples", [])
        
        if not examples:
            continue
        
        ground_truth = [float(ex.get("output", 0)) for ex in examples]
        our_preds = [float(ex.get("predict_our_method", 0)) for ex in examples]
        baseline_preds = [float(ex.get("predict_baseline", 0)) for ex in examples]
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle(f'Results for {dataset_name}', fontsize=14)
        
        axes[0].scatter(ground_truth, our_preds, alpha=0.6, s=100, color='blue', label='Our Method')
        axes[0].plot([min(ground_truth), max(ground_truth)], [min(ground_truth), max(ground_truth)], 'r--', alpha=0.5, label='Perfect')
        axes[0].set_xlabel('Ground Truth (Grade Level)')
        axes[0].set_ylabel('Predicted (Resistance Score)')
        axes[0].set_title('Our Method: Graph Resistance')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        axes[1].scatter(ground_truth, baseline_preds, alpha=0.6, s=100, color='green', label='Baseline')
        axes[1].plot([min(ground_truth), max(ground_truth)], [min(ground_truth), max(ground_truth)], 'r--', alpha=0.5, label='Perfect')
        axes[1].set_xlabel('Ground Truth (Grade Level)')
        axes[1].set_ylabel('Predicted (FK Grade)')
        axes[1].set_title('Baseline: Flesch-Kincaid')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print(f"\n{'='*80}")
        print(f"Detailed Results for {dataset_name}")
        print(f"{'='*80}")
        print(f"{'#':<4} {'Ground Truth':<15} {'Our Method':<15} {'Baseline (FK)':<15} {'Text Preview'}")
        print(f"{'-'*80}")
        
        for i, ex in enumerate(examples):
            gt = float(ex.get('output', 0))
            our = float(ex.get('predict_our_method', 0))
            base = float(ex.get('predict_baseline', 0))
            text_preview = ex.get('input', '')[:40].replace('\n', ' ') + '...'
            print(f"{i+1:<4} {gt:<15.2f} {our:<15.4f} {base:<15.2f} {text_preview}")


visualize_results(results)

In [ ]:
# Save results to JSON
output_path = "demo_results.json"
with open(output_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {output_path}")

print("\n" + "="*60)
print("EXPERIMENT COMPLETED SUCCESSFULLY!")
print("="*60)
print("\nKey findings:")
print("- Graph resistance method computes novel readability scores")
print("- Flesch-Kincaid provides traditional baseline")
print("- Compare MAE/RMSE values to evaluate method performance")
print("\nNext steps:")
print("- Scale up to larger dataset (adjust MAX_EXAMPLES)")
print("- Tune SIMILARITY_THRESHOLD parameter")
print("- Experiment with different graph construction methods")